In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


X = pd.read_csv('../data/X_train_week4.csv')
y = pd.read_csv('../data/y_train_week4.csv')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


models = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1)
}


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nCross-Validation F1 Scores:")
for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train.values.ravel(),
        cv=skf,
        scoring='f1'
    )
    print(f"{name}: {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")


param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': ['balanced', 'balanced_subsample']
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train.values.ravel())
best_model = grid.best_estimator_

print("\nBest Random Forest Parameters:")
print(grid.best_params_)
print(f"Best CV F1-score: {grid.best_score_ * 100:.2f}%")


y_val_pred = best_model.predict(X_val)

accuracy = accuracy_score(y_val, y_val_pred) * 100
precision = precision_score(y_val, y_val_pred) * 100
recall = recall_score(y_val, y_val_pred) * 100
f1 = f1_score(y_val, y_val_pred) * 100

print("\nValidation Metrics:")
print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1-score : {f1:.2f}%")


model_dir = "../flask-backend/model"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "final_model_week5.pkl")
columns_path = os.path.join(model_dir, "model_columns.pkl")

with open(model_path, "wb") as f:
    pickle.dump(best_model, f)

with open(columns_path, "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)

print("\nModel saved at:", model_path)
print("Model columns saved at:", columns_path)


with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

print("\nLoaded model type:", type(loaded_model))
print("Predict method exists:", hasattr(loaded_model, "predict"))



Cross-Validation F1 Scores:
DecisionTree: 33.26% ± 1.39%
RandomForest: 20.95% ± 2.38%

Best Random Forest Parameters:
{'class_weight': 'balanced_subsample', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Best CV F1-score: 44.00%

Validation Metrics:
Accuracy : 44.25%
Precision: 29.39%
Recall   : 90.47%
F1-score : 44.36%

Model saved at: ../flask-backend/model\final_model_week5.pkl
Model columns saved at: ../flask-backend/model\model_columns.pkl

Loaded model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Predict method exists: True
